In [13]:
!pip install typing_extensions --upgrade
!pip install matplotlib
!pip install matplotlib
!pip install scikit-learn
!pip install timm

In [14]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install numpy matplotlib scikit-learn pillow timm

Looking in indexes: https://download.pytorch.org/whl/cpu


In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import itertools
import os
import time
from PIL import Image

In [25]:
# হাইপারপ্যারামিটার
IMAGE_SIZE = 300  # TinyViT-21M 224x224 ইনপুট নেয়
BATCH_SIZE = 16
CHANNELS = 3
EPOCHS = 30
DATA_DIR = r'E:\4.2\4.1\Thesis\Data\Healthy and Unhealthy Papaya Leaf Images from Bangladeshi Orchards\Original Dataset'
DEVICE = torch.device("cpu")

In [26]:
# ডেটা অগমেন্টেশন এবং প্রি-প্রসেসিং
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        transforms.Lambda(lambda x: x + torch.randn_like(x) * 0.1)
    ]),
    'val': transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

In [27]:
# কাস্টম ডেটাসেট ক্লাস
class CustomImageDataset(datasets.ImageFolder):
    def __init__(self, root, transform=None):
        super(CustomImageDataset, self).__init__(root, transform)
    
    def __getitem__(self, index):
        path, target = self.samples[index]
        image = Image.open(path).convert('RGB')  # সব ইমেজ RGB তে কনভার্ট
        if self.transform is not None:
            image = self.transform(image)
        return image, target

In [28]:
# ডেটাসেট লোড
full_dataset = CustomImageDataset(DATA_DIR, transform=data_transforms['train'])
class_names = full_dataset.classes
n_classes = len(class_names)

def dataset_split(dataset):
    size = len(dataset)
    indices = list(range(size))
    np.random.seed(42)
    np.random.shuffle(indices)
    train_end = int(0.75 * size)
    val_end = train_end + int(0.15 * size)
    return torch.utils.data.Subset(dataset, indices[:train_end]), \
           torch.utils.data.Subset(dataset, indices[train_end:val_end]), \
           torch.utils.data.Subset(dataset, indices[val_end:])

train_dataset, val_dataset, test_dataset = dataset_split(full_dataset)
train_dataset.dataset.transform = data_transforms['train']
val_dataset.dataset.transform = data_transforms['val']
test_dataset.dataset.transform = data_transforms['test']

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [30]:
model = models.mobilenet_v3_small(pretrained=True)
in_features = model.classifier[0].in_features  # 576 for small variant

model.classifier = nn.Sequential(
    nn.Linear(in_features, 256),
    nn.ReLU(),
    nn.Dropout(p=0.8),
    nn.Linear(256, n_classes)
)

model.to(DEVICE)


# লস ও অপ্টিমাইজার
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [32]:
# ট্রেনিং ফাংশন
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs, device):
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(epochs):
        model.train()
        correct, total, running_loss = 0, 0, 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
        train_acc = correct / total
        train_loss = running_loss / len(train_loader)
        
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                correct += (predicted == labels).sum().item()
                total += labels.size(0)
        val_acc = correct / total
        val_loss = val_loss / len(val_loader)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f"Epoch [{epoch+1}/{epochs}] -> Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    return history



In [33]:
history = train_model(model, train_loader, val_loader, criterion, optimizer, EPOCHS, DEVICE)

Epoch [1/30] -> Train Loss: 1.6919, Train Acc: 0.4355, Val Loss: 22.5971, Val Acc: 0.1723
Epoch [2/30] -> Train Loss: 1.6126, Train Acc: 0.4883, Val Loss: 13.1197, Val Acc: 0.1765


KeyboardInterrupt: 

In [ ]:
# ট্রেনিং প্লট
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['val_acc'], label='Val Acc')
plt.title('Accuracy')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# টেস্ট সেটে পারফরম্যান্স
model.eval()
y_true, y_pred, y_probs = [], [], []
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(probs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        y_probs.extend(probs.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_probs = np.array(y_probs)

print("\n📊 Test Set Metrics")
print(classification_report(y_true, y_pred, target_names=class_names))
print(f"✅ Test Accuracy: {accuracy_score(y_true, y_pred):.4f}")

In [ ]:
# Confusion Matrix
conf_matrix = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
plt.imshow(conf_matrix, cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()
ticks = np.arange(len(class_names))
plt.xticks(ticks, class_names, rotation=45)
plt.yticks(ticks, class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Save model
torch.save(model.state_dict(), 'mobilenetv3_final.pth')
model_size = os.path.getsize('mobilenetv3_final.pth') / (1024 * 1024)
print(f"Model Size: {model_size:.2f} MB")

# Inference speed
dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
times = []
with torch.no_grad():
    for _ in range(10):
        start = time.time()
        _ = model(dummy_input)
        times.append((time.time() - start) * 1000)
print(f"Average Inference Time: {np.mean(times):.2f} ms per image")

In [ ]:
# ট্রেনিং এবং ভ্যালিডেশন প্লট
plt.figure(figsize=(8, 8))
plt.subplot(1, 2, 1)
plt.plot(history['train_acc'], label='Training Accuracy')
plt.plot(history['val_acc'], label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')
plt.subplot(1, 2, 2)
plt.plot(history['train_loss'], label='Training Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
# ভবিষ্যদ্বাণী ফাংশন
def predict(model, image, device):
    model.eval()
    image = image.unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(image)
        probabilities = torch.softmax(output, dim=1)
        confidence, predicted = torch.max(probabilities, 1)
    return class_names[predicted.item()], confidence.item() * 100

# টেস্ট ডেটাসেটে ভবিষ্যদ্বাণী
plt.figure(figsize=(15, 15))
for images, labels in test_loader:
    for i in range(min(9, len(images))):
        plt.subplot(3, 3, i + 1)
        predicted_class, confidence = predict(model, images[i], DEVICE)
        actual_class = class_names[labels[i]]
        imshow(images[i], f'Actual: {actual_class}\nPredicted: {predicted_class}\nConfidence: {confidence:.2f}%')
    break
plt.tight_layout()
plt.show()

In [ ]:
# মেট্রিক্স গণনা
y_true = []
y_pred = []
y_probs = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        probabilities = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())
        y_probs.extend(probabilities.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_probs = np.array(y_probs)

In [ ]:
# মেট্রিক্স প্রিন্ট
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

overall_accuracy = accuracy_score(y_true, y_pred)
overall_precision = precision_score(y_true, y_pred, average='weighted')
overall_recall = recall_score(y_true, y_pred, average='weighted')
overall_f1 = f1_score(y_true, y_pred, average='weighted')
roc_auc = roc_auc_score(np.eye(n_classes)[y_true], y_probs, average='weighted', multi_class='ovr')

print(f"✅ Overall Accuracy: {overall_accuracy:.4f}")
print(f"✅ Overall Precision: {overall_precision:.4f}")
print(f"✅ Overall Recall: {overall_recall:.4f}")
print(f"✅ Overall F1-Score: {overall_f1:.4f}")
print(f"✅ Overall ROC AUC Score: {roc_auc:.4f}")

In [ ]:
# কনফিউশন ম্যাট্রিক্স
def plot_confusion_matrix(cm, classes, normalize=False, title='Confusion Matrix', cmap=plt.cm.Blues):
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')
    
    print(cm)
    
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)
    
    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")
    
    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

conf_matrix = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
plot_confusion_matrix(conf_matrix, class_names, title='Confusion Matrix')
plt.show()
plt.figure(figsize=(8, 6))
plot_confusion_matrix(conf_matrix, class_names, normalize=True, title='Normalized Confusion Matrix')
plt.show()

In [ ]:
# মডেল সাইজ এবং ইনফারেন্স স্পিড
torch.save(model.state_dict(), 'mobilenetv3_small_model.pth')
model_size_mb = os.path.getsize('mobilenetv3_small_model.pth') / (1024 * 1024)
print(f"Model Size: {model_size_mb:.2f} MB")

dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
times = []
model.eval()
with torch.no_grad():
    for _ in range(10):
        start_time = time.time()
        _ = model(dummy_input)
        end_time = time.time()
        times.append((end_time - start_time) * 1000)

avg_inference_time = sum(times) / len(times)
print(f"Average Inference Time: {avg_inference_time:.2f} ms per image")